In [1]:
from stmpy import Driver, Machine


class QuizMachine:
    def __init__(self):
        self.mqtt_client = None
        self.stm = None

    def on_init(self):
        print("Quiz system initialized. Waiting for questions...")

    def send_mqtt_question(self):
        print("Question asked! 20 second timer started.")
        self.mqtt_client.publish(topic="quiz/question", payload="Question asked - Buzzers active!")

    def on_buzz(self, participant_id):
        print(f"BUZZ! {participant_id} pressed first!")
        self.mqtt_client.publish(topic="quiz/result", payload=f"Winner: {participant_id}")
        self.stm.stop_timer("t")

    def on_timeout(self):
        print("Time's up! No one buzzed in time.")
        self.mqtt_client.publish(topic="quiz/result", payload="Timeout - no answer")

    def reset_quiz(self):
        print("Quiz reset. Ready for next question.")
        self.mqtt_client.publish(topic="quiz/status", payload="Ready for next question")


# States
idle = {"name": "s_idle"}
buzzwait = {
    "name": "s_buzzwait", 
    "entry": 'start_timer("t", 20000); send_mqtt_question()'
}

# Transitions
t0 = {"source": "initial", "target": "s_idle", "effect": "on_init"}

# Question asked - start timer
t1 = {
    "trigger": "question",
    "source": "s_idle",
    "target": "s_buzzwait"
}

# Someone buzzed - return to idle
t2 = {
    "trigger": "buzz",
    "source": "s_buzzwait",
    "target": "s_idle",
    "effect": "on_buzz(*)"
}

# Timer expired - return to idle
t3 = {
    "trigger": "t",
    "source": "s_buzzwait",
    "target": "s_idle",
    "effect": "on_timeout"
}

# Reset from any state
t4 = {
    "trigger": "reset",
    "source": "s_idle",
    "target": "s_idle",
    "effect": "reset_quiz"
}

t5 = {
    "trigger": "reset",
    "source": "s_buzzwait",
    "target": "s_idle",
    "effect": "reset_quiz"
}

In [2]:
from threading import Thread
import paho.mqtt.client as mqtt


class MQTT_Client:
    def __init__(self):
        self.client = mqtt.Client()
        self.client.on_connect = self.on_connect
        self.client.on_message = self.on_message
        self.stm_driver = None

    def on_connect(self, client, userdata, flags, rc):
        print("Connected to MQTT broker: {}".format(mqtt.connack_string(rc)))

    def on_message(self, client, userdata, msg):
        topic = msg.topic
        payload = msg.payload.decode('utf-8')
        print(f"Received: {topic} -> {payload}")

        if topic == "quiz/start":
            # Quiz master starts question
            self.stm_driver.send("question", "quiz_machine")
        elif topic.startswith("quiz/buzz/"):
            # Participant buzzes (e.g., quiz/buzz/participant1)
            participant_id = topic.split("/")[-1]
            self.stm_driver.send("buzz", "quiz_machine", args=[participant_id])
        elif topic == "quiz/reset":
            # Reset the quiz
            self.stm_driver.send("reset", "quiz_machine")

    def start(self, broker, port):
        print(f"Connecting to {broker}:{port}")
        self.client.connect(broker, port)

        # Subscribe to relevant topics
        self.client.subscribe("quiz/start")
        self.client.subscribe("quiz/buzz/#")
        self.client.subscribe("quiz/reset")
        
        print("Instructions:")
        print("  - Send to 'quiz/start' to ask a question")
        print("  - Send to 'quiz/buzz/participant1' (or participant2, etc.) to buzz")
        print("  - Send to 'quiz/reset' to reset the quiz\n")

        try:
            thread = Thread(target=self.client.loop_forever)
            thread.start()
        except KeyboardInterrupt:
            print("Interrupted")
            self.client.disconnect()

In [ ]:
# broker, port = 'iot.eclipse.org', 1883
broker, port = "mqtt20.item.ntnu.no", 1883

# Create quiz machine
quiz = QuizMachine()
quiz_machine = Machine(
    transitions=[t0, t1, t2, t3, t4, t5], 
    obj=quiz, 
    name="quiz_machine",
    states=[idle, buzzwait]
)
quiz.stm = quiz_machine

# Create driver and add machine
driver = Driver()
driver.add_machine(quiz_machine)

# Create MQTT client and link everything
mqtt_client = MQTT_Client()
quiz.mqtt_client = mqtt_client.client
mqtt_client.stm_driver = driver

# Start the system
driver.start()
mqtt_client.start(broker, port)

Quiz system initialized. Waiting for questions...
Connecting to mqtt20.item.ntnu.no:1883
Instructions:
  - Send to 'quiz/start' to ask a question
  - Send to 'quiz/buzz/participant1' (or participant2, etc.) to buzz
  - Send to 'quiz/reset' to reset the quiz



/tmp/ipykernel_42080/3108565299.py:7: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  self.client = mqtt.Client()


Connected to MQTT broker: Connection Accepted.


Received: quiz/start -> {
  "msg": "Where is the blackberry PI"
}
Question asked! 20 second timer started.
Received: quiz/buzz/participant1 -> Correct answer
BUZZ! participant1 pressed first!
Connected to MQTT broker: Connection Accepted.
Connected to MQTT broker: Connection Accepted.
Connected to MQTT broker: Connection Accepted.
Connected to MQTT broker: Connection Accepted.
